**Instalação e Importação das Bibliotecas**

In [ ]:
# Instale o kagglehub: pip install kagglehub

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("Versão do TensorFlow:", tf.__version__)

**Download e Carregamento dos Dados**

Usaremos os arquivos do PTB Diagnostic ECG Database inclusos no dataset.

In [ ]:
# Download da versão mais recente do dataset
path = kagglehub.dataset_download("shayanfazeli/heartbeat")
print("Caminho dos arquivos:", path)

# Carregando os CSVs específicos para classificação binária (PTBDB)
df_normal = pd.read_csv(f"{path}/ptbdb_normal.csv", header=None)
df_abnormal = pd.read_csv(f"{path}/ptbdb_abnormal.csv", header=None)

# Adicionando as labels (0 para Normal, 1 para Anormal)
df_normal['label'] = 0
df_abnormal['label'] = 1

# Unindo os dataframes e embaralhando os dados
df_completo = pd.concat([df_normal, df_abnormal], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

# Pegando apenas as 187 colunas do sinal de ECG
X_raw = df_completo.iloc[:, :187].values
y = df_completo['label'].values

print(f"Formato de X_raw: {X_raw.shape} (Deve ser N linhas por 187 colunas)")

**Transformando o dataset em Imagem**

In [ ]:
# O sinal original tem 187 pontos. Para transformá-lo em uma "imagem" quadrada,
# precisamos de um quadrado perfeito. O mais próximo é 14x14 = 196.
# Faremos um padding (preenchimento com zeros) de 9 posições no final de cada amostra.

X_padded = np.pad(X_raw, ((0, 0), (0, 9)), mode='constant', constant_values=0)

# Reshape para formato de imagem 2D (14x14) - Simulando uma imagem em tons de cinza
X_images = X_padded.reshape(-1, 14, 14)

print("Formato após redimensionamento para imagens 2D:", X_images.shape)

# Visualizando um exemplo transformado em imagem
plt.figure(figsize=(4, 4))
plt.imshow(X_images[0], cmap='gray')
plt.title(f"ECG Transformado em Matriz 14x14\nLabel: {'Anormal' if y[0] == 1 else 'Normal'}")
plt.axis('off')
plt.show()

# Dividindo em treino (80%) e teste (20%)
X_train, X_test, y_train, y_test = train_test_split(X_images, y, test_size=0.2, random_state=42, stratify=y)

**Criação da Rede Neural MLP**

Redes MLP (Multilayer Perceptrons) exigem entradas unidimensionais. A primeira camada da rede será uma camada Flatten para achatar nossa imagem 14x14 de volta para um vetor, unindo o conceito de processamento visual com a arquitetura exigida.

In [ ]:
# Construindo o modelo MLP com a sintaxe mais recente do Keras
model = keras.Sequential([
    # Nova forma recomendada de declarar a entrada (evita o UserWarning)
    keras.Input(shape=(14, 14), name="Entrada_Dados"),

    # Camada que achata a matriz 14x14 para um vetor de 196 posições
    layers.Flatten(name="Flatten_Imagem"),

    # Primeira camada oculta com 128 neurônios e ativação ReLU
    layers.Dense(128, activation='relu', name="Oculta_1"),
    # Dropout para evitar overfitting
    layers.Dropout(0.3),

    # Segunda camada oculta com 64 neurônios
    layers.Dense(64, activation='relu', name="Oculta_2"),
    layers.Dropout(0.2),

    # Camada de saída com 1 neurônio e ativação Sigmoid (ideal para classificação binária)
    layers.Dense(1, activation='sigmoid', name="Saida_Binaria")
])

# Compilando o modelo
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

**Treinamento do Modelo**

In [ ]:
# Treinando o modelo
# Usamos validation_split para monitorar o desempenho em dados não vistos durante o treino
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

**Avaliação e Resultados**

In [ ]:
# Avaliando a acurácia no conjunto de teste
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nAcurácia no conjunto de Teste: {test_acc:.4f}")

# Gerando previsões (probabilidades) e convertendo para classes (0 ou 1)
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype("int32")

# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Anormal'], yticklabels=['Normal', 'Anormal'])
plt.title('Matriz de Confusão')
plt.ylabel('Rótulo Verdadeiro')
plt.xlabel('Previsão do Modelo')
plt.show()

# Relatório de Classificação (Precision, Recall, F1-Score)
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Anormal']))